In [19]:
import numpy as np


def build_2D_system(p0, p1, p2):
    # Local x-axis along p1 - p0
    x_local = p1 - p0
    x_len = np.linalg.norm(x_local, axis=1)[:, np.newaxis]
    x_local /= x_len

    # Normal vector of the plane
    n = np.cross(p1 - p0, p2 - p0)
    n_len = np.linalg.norm(n, axis=1)[:, np.newaxis]
    n /= n_len

    # Local y-axis
    y_local = np.cross(n, x_local)

    return x_local, y_local, n

def project_to_plane(p0, pi, x_local, y_local):
    v = pi - p0
    return np.stack([np.einsum('ij,ij->i', v, x_local),
                     np.einsum('ij,ij->i', v, y_local)], axis=1)



def volumes_and_grads(self, coords, elems):
    """
    coords: (N_nodes, 3)
    elems: (N_elems, 3)
    Returns:
        areas: (N_elems,)
        grads: (N_elems, 3, 3)
    """
    N = elems.shape[0]
    grads = np.zeros((N, 3, 3))
    areas = np.zeros(N)

    # Node coordinates
    p0 = coords[elems[:, 0]]
    p1 = coords[elems[:, 1]]
    p2 = coords[elems[:, 2]]

    # Local x-axis along p1 - p0
    x_local, y_local, n = build_2D_system(p0, p1, p2)

    p0_2d = np.zeros((N, 2))
    p1_2d = project_to_plane(p0, p1, x_local, y_local)
    p2_2d = project_to_plane(p0, p2, x_local, y_local)

    # Compute triangle area in 2D
    v01 = p1_2d - p0_2d
    v02 = p2_2d - p0_2d
    areas = 0.5 * np.abs(v01[:, 0] * v02[:, 1] - v01[:, 1] * v02[:, 0])

    # Compute gradients in 2D
    # For linear triangle: grad(N0) = cross(edge opposite)/2A, etc.
    grads_local = np.zeros((N, 3, 2))
    # Edge vectors
    v = np.array([p1_2d - p2_2d, p2_2d - p0_2d, p0_2d - p1_2d])
    for i in range(3):
        grads_local[:, i, 0] =  v[i][:, 1] / (2 * areas)
        grads_local[:, i, 1] = -v[i][:, 0] / (2 * areas)

    # Map gradients back to 3D
    for i in range(3):
        grads[:, i, :] = (grads_local[:, i, 0][:, np.newaxis] * x_local + 
                          grads_local[:, i, 1][:, np.newaxis] * y_local)

    return areas, grads


def volumes_and_grads_old(coords, elems):
    grads = np.zeros((elems.shape[0], 3, 3))

    # vertice 0
    p0 = coords[elems[:, 0]]
    p1 = coords[elems[:, 1]]
    p2 = coords[elems[:, 2]]

    v0 = p1 - p0
    v1 = p2 - p0

    normals = np.cross(v0, v1)
    areas = np.linalg.norm(normals, axis=1) / 2.0

    grads[:, 0, :] = (np.cross(p1 - p2, normals) /
                        (2.0 * areas[:, np.newaxis]) ** 2)
    grads[:, 1, :] = (np.cross(p2 - p0, normals) /
                        (2.0 * areas[:, np.newaxis]) ** 2)
    grads[:, 2, :] = (np.cross(p0 - p1, normals) /
                        (2.0 * areas[:, np.newaxis]) ** 2)

    return areas, grads


# coords = np.array([[0.0, 0.0, 0.0],
#                    [1.0, 0.0, 0.0],
#                    [0.0, 1.0, 1.0]])

coords = np.random.rand(3, 3)
elems = np.array([[0, 1, 2]])

areas_old, grads_old = volumes_and_grads_old(coords, elems)
print("Old Areas:", areas_old)
print("Old Grads:", grads_old)

areas, grads = volumes_and_grads(None, coords, elems)
print("Areas:", areas)
print("Grads:", grads)



Old Areas: [0.31715604]
Old Grads: [[[ 1.3523012   0.0591903  -0.17348194]
  [-1.33187733 -0.16643797 -0.98811132]
  [-0.02042388  0.10724767  1.16159326]]]
Areas: [0.31715604]
Grads: [[[ 1.3523012   0.0591903  -0.17348194]
  [-1.33187733 -0.16643797 -0.98811132]
  [-0.02042388  0.10724767  1.16159326]]]


In [23]:
import numpy as np


def volumes_and_grads_jacobian(coords, elems):
    """
    Computes area and global gradients for linear triangles in 3D
    using the Jacobian from the reference triangle.

    coords: (N_nodes, 3)
    elems: (N_elems, 3)
    Returns:
        areas: (N_elems,)
        grads: (N_elems, 3, 3)
    """
    # Node coordinates
    p0 = coords[elems[:, 0]]
    p1 = coords[elems[:, 1]]
    p2 = coords[elems[:, 2]]

    # Jacobian (2x3 per element)
    J = np.stack([p1 - p0, p2 - p0], axis=1)  # shape (N, 2, 3)

    # Area = 0.5 * |cross(p1 - p0, p2 - p0)|
    cross_v = np.cross(p1 - p0, p2 - p0)
    areas = 0.5 * np.linalg.norm(cross_v, axis=1)

    # Local derivatives (constant for linear triangle)
    dN_dxi = np.array([-1.0, 1.0, 0.0])
    dN_deta = np.array([-1.0, 0.0, 1.0])

    # Compute J * grad(N_ref)^T = mapping to 3D
    # First get the 2x2 metric tensor: G = J * J^T
    # But since J is 2x3, we use the pseudoinverse.
    JT = np.transpose(J, (0, 2, 1))  # (N, 3, 2)
    G = np.matmul(J, JT)             # (N, 2, 2)
    invG = np.linalg.inv(G)
    Jplus = np.matmul(JT, invG)      # (N, 3, 2) — right pseudoinverse

    # Gradients in global coordinates
    grads = np.zeros((elems.shape[0], 3, 3))
    for i in range(3):
        dN_ref = np.stack([np.full(elems.shape[0], dN_dxi[i]),
                           np.full(elems.shape[0], dN_deta[i])], axis=1)
        grads[:, i, :] = np.einsum('nij,nj->ni', Jplus, dN_ref)

    return areas, grads


volumes_and_grads_jacobian(coords, elems)

(array([0.31715604]),
 array([[[ 1.3523012 ,  0.0591903 , -0.17348194],
         [-1.33187733, -0.16643797, -0.98811132],
         [-0.02042388,  0.10724767,  1.16159326]]]))

In [26]:
import numpy as np


def volumes_and_grads_quad(coords, elems):
    """
    Computes area and global gradients for bilinear quadrilaterals in 3D
    using the Jacobian from the reference element at its center.

    coords: (N_nodes, 3)
    elems: (N_elems, 4)
    Returns:
        areas: (N_elems,)
        grads: (N_elems, 4, 3)
    """
    N = elems.shape[0]
    grads = np.zeros((N, 4, 3))
    areas = np.zeros(N)

    # Node coordinates
    p0 = coords[elems[:, 0]]
    p1 = coords[elems[:, 1]]
    p2 = coords[elems[:, 2]]
    p3 = coords[elems[:, 3]]

    # Derivatives of shape functions at element center
    dN_dxi  = np.array([-0.25,  0.25,  0.25, -0.25])
    dN_deta = np.array([-0.25, -0.25,  0.25,  0.25])

    # Jacobian J = [∂x/∂ξ  ∂x/∂η] (2×3 per element)
    J = np.zeros((N, 2, 3))
    for i in range(4):
        J[:, 0, :] += dN_dxi[i]  * coords[elems[:, i]]
        J[:, 1, :] += dN_deta[i] * coords[elems[:, i]]

    # Compute normal and area via cross product of the Jacobian rows
    v_xi  = J[:, 0, :]
    v_eta = J[:, 1, :]
    cross_v = np.cross(v_xi, v_eta)
    areas = np.linalg.norm(cross_v, axis=1)

    # Right pseudoinverse of J: J⁺ = Jᵀ (J Jᵀ)⁻¹
    JT = np.transpose(J, (0, 2, 1))  # (N, 3, 2)
    G = np.matmul(J, JT)             # (N, 2, 2)
    invG = np.linalg.inv(G)
    Jplus = np.matmul(JT, invG)      # (N, 3, 2)

    # Compute global gradients for each node
    for i in range(4):
        dN_ref = np.stack([np.full(N, dN_dxi[i]),
                            np.full(N, dN_deta[i])], axis=1)
        grads[:, i, :] = np.einsum('nij,nj->ni', Jplus, dN_ref)

    return areas, grads


q_coords = np.array([[0, 0, 0],
                     [0, 2, 0],
                     [1, 1, 0],
                     [1, 0, 0]])

q_elems = np.array([[0, 1, 2, 3]])

a, g =volumes_and_grads_quad(q_coords, q_elems)
print(a * 4)

[1.5]
